<a href="https://colab.research.google.com/github/nawroz-m/ML_learning/blob/BigImgSingnal/titanic_random_forest_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import sklearn


In [ ]:
# load dataset
drive.flush_and_unmount()
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/Big Img Sig/titanic.csv"
pdf = pd.read_csv(path)

In [ ]:
pdf.shape

(891, 12)

In [ ]:
pdf.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [ ]:
pdf[:10]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [ ]:
pdf.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [ ]:
(pdf.isna().sum()/len(pdf)) * 100

,0
PassengerId,0.000000
Survived,0.000000
Pclass,0.000000
Name,0.000000
Sex,0.000000
Age,19.865320
SibSp,0.000000
Parch,0.000000
Ticket,0.000000
Fare,0.000000


In [ ]:
pdf[:4]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


## clean the dataset

In [ ]:
# Drop coluns 'Name', Ticket', "Cabin"

In [ ]:
# conver to boolean type
pdf = pdf.convert_dtypes()
pdf.dtypes

,0
PassengerId,Int64
Survived,Int64
Pclass,Int64
Name,string[python]
Sex,string[python]
Age,Float64
SibSp,Int64
Parch,Int64
Ticket,string[python]
Fare,Float64


In [ ]:
# Drop these columns since they are not necessary to train the model on
drop_columns = ["Name", "Ticket", "Cabin"]
pdf = pdf.drop(drop_columns, axis=1)
pdf

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.25,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.925,S
3,4,1,1,female,35.0,1,0,53.1,S
4,5,0,3,male,35.0,0,0,8.05,S
...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.0,0,0,13.0,S
887,888,1,1,female,19.0,0,0,30.0,S
888,889,0,3,female,<NA>,1,2,23.45,S
889,890,1,1,male,26.0,0,0,30.0,C


In [ ]:
pdf[:10]

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.25,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.925,S
3,4,1,1,female,35.0,1,0,53.1,S
4,5,0,3,male,35.0,0,0,8.05,S
5,6,0,3,male,<NA>,0,0,8.4583,Q
6,7,0,1,male,54.0,0,0,51.8625,S
7,8,0,3,male,2.0,3,1,21.075,S
8,9,1,3,female,27.0,0,2,11.1333,S
9,10,1,2,female,14.0,1,0,30.0708,C


In [ ]:
# check if we have any boolean data
pdf.select_dtypes(include=bool).columns

Index([], dtype='object')

In [ ]:
# check the empty dataset
pdf.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [ ]:
# select numeric columns to clean up
numeric = pdf.select_dtypes(include=int)
numeric_column = numeric.columns
numeric_column

Index(['PassengerId', 'Survived', 'Pclass', 'SibSp', 'Parch'], dtype='object')

In [ ]:
# Find the median of numeric columns
numeric_median = pdf[numeric_column].median()
numeric_median

,0
PassengerId,446.0
Survived,0.0
Pclass,3.0
SibSp,0.0
Parch,0.0


In [ ]:
# Replace those numerical columns missing values with the median of these column
pdf[numeric_column] = pdf[numeric_column].fillna(numeric_median)
pdf[:10]


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.25,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.925,S
3,4,1,1,female,35.0,1,0,53.1,S
4,5,0,3,male,35.0,0,0,8.05,S
5,6,0,3,male,<NA>,0,0,8.4583,Q
6,7,0,1,male,54.0,0,0,51.8625,S
7,8,0,3,male,2.0,3,1,21.075,S
8,9,1,3,female,27.0,0,2,11.1333,S
9,10,1,2,female,14.0,1,0,30.0708,C


In [ ]:
# find all float data type
float_columns = pdf.select_dtypes(include=float).columns
float_columns

Index(['Age', 'Fare'], dtype='object')

In [ ]:
# get the medaian of the float
float_median = pdf[float_columns].median()
float_median

,0
Age,28.0
Fare,14.4542


In [ ]:
# replace the empty dataset of float
pdf[float_columns] = pdf[float_columns].fillna(float_median)
pdf

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.25,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.925,S
3,4,1,1,female,35.0,1,0,53.1,S
4,5,0,3,male,35.0,0,0,8.05,S
...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.0,0,0,13.0,S
887,888,1,1,female,19.0,0,0,30.0,S
888,889,0,3,female,28.0,1,2,23.45,S
889,890,1,1,male,26.0,0,0,30.0,C


In [ ]:
pdf.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,2


In [ ]:
pdf.select_dtypes(include="string").isna().sum()

,0
Sex,0
Embarked,2


In [ ]:
# Get the categorical columns
categorical_column = pdf.select_dtypes(include='string').columns.tolist()
categorical_column

['Sex', 'Embarked']

In [ ]:
# get the most frequent vlaues of categorical columns
categorical_column_freq = pdf[categorical_column].mode().loc[0]
categorical_column_freq

,0
Sex,male
Embarked,S


In [ ]:
# Replace the categorical columns with most frequent values
pdf[categorical_column] = pdf[categorical_column].fillna(categorical_column_freq)
pdf.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [ ]:
# Final check
pdf.isnull().values.any()

np.False_

## Feature encoding

In [ ]:
# 1. Label encoding
label_encoder = sklearn.preprocessing.LabelEncoder()

In [ ]:
# Encode the ground throught
pdf['Embarked'] = label_encoder.fit_transform(pdf['Embarked'])
pdf[:5]

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.25,2
1,2,1,1,female,38.0,1,0,71.2833,0
2,3,1,3,female,26.0,0,0,7.925,2
3,4,1,1,female,35.0,1,0,53.1,2
4,5,0,3,male,35.0,0,0,8.05,2


**Encode categorical feature through one-hot-encoder**


In [ ]:
# Get the the categorical features
categorical_feature = pdf.select_dtypes(include='string').columns.tolist()
categorical_feature

['Sex']

In [ ]:
# One-hot encode the categorical features
pdf = pd.get_dummies(pdf, prefix=categorical_feature)

In [ ]:
pdf[:3]

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked,Sex_female,Sex_male
0,1,0,3,22.0,1,0,7.25,2,False,True
1,2,1,1,38.0,1,0,71.2833,0,True,False
2,3,1,3,26.0,0,0,7.925,2,True,False


**Find the outliers**

In [ ]:
# Initialize the random staste
isolution_forest = sklearn.ensemble.IsolationForest(n_estimators=1000, contamination=0.01, random_state=0)

In [ ]:
# Get anomely scor
res = isolution_forest.fit_predict(pdf.to_numpy())

In [ ]:
res[res==-1]

array([-1, -1, -1, -1, -1, -1, -1, -1, -1])

In [ ]:
# Find outliers with PCA
pca = sklearn.decomposition.PCA(n_components=0.9999)
pca

PCA(n_components=0.9999)

In [ ]:
# apply pca
Xpca = pca.fit_transform(pdf)
Xpca[:5]

array([[-445.07417859,  -24.00147188,   -5.94838451],
       [-443.88371966,   40.46571305,    8.32834914],
       [-443.06544484,  -23.22125732,   -1.9628525 ],
       [-441.9350576 ,   22.1968872 ,    5.82023556],
       [-441.0493295 ,  -22.86409086,    7.03346444]])

In [ ]:
# Inverce the PCA
X_ori = pca.inverse_transform(Xpca)
X_ori[:2]

array([[ 0.99998779,  0.34808224,  2.69157835, 21.99578673,  0.67530353,
         0.37311721,  7.24350983,  1.6032475 ,  0.36763033,  0.63236967],
       [ 1.99979708,  0.46100724,  1.83174929, 38.00588373,  0.59669793,
         0.42259571, 71.29909238,  1.37569963,  0.42806776,  0.57193224]])

In [ ]:
# Define anomaly score as the sum of absolute distance
anomaly_distance = np.abs(pdf.to_numpy()-X_ori)
anomaly_distance[:2]

array([[1.2212070885198045e-05, 0.34808224062486803, 0.3084216476879771,
        0.004213269552749921, 0.32469647306803406, 0.3731172111234562,
        0.006490171369357256, 0.3967524971880474, 0.36763033304097464,
        0.36763033304097514],
       [0.00020292141147137954, 0.5389927565964547, 0.8317492942751223,
        0.00588373077840032, 0.4033020694098479, 0.42259571400535123,
        0.01579238309415132, 1.3756996292344597, 0.5719322422570692,
        0.57193224225707]], dtype=object)

In [ ]:
anomaly_score = anomaly_distance.sum(1)
anomaly_score[:4]

array([2.497046388767325, 4.738082983319398, 3.669024276549157,
       4.196959727581794], dtype=object)

In [ ]:
# define the threshold
threshold = np.quantile(anomaly_score, 0.99)
threshold

np.float64(9.740533241505814)

In [ ]:
# get the ids of anomalous values
anomalous_id = np.argwhere(anomaly_score>threshold).squeeze()
anomalous_id

array([159, 180, 201, 324, 678, 737, 792, 846, 863])

In [ ]:
pdf.iloc[anomalous_id]

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked,Sex_female,Sex_male
159,160,0,3,28.0,8,2,69.55,2,False,True
180,181,0,3,28.0,8,2,69.55,2,True,False
201,202,0,3,28.0,8,2,69.55,2,False,True
324,325,0,3,28.0,8,2,69.55,2,False,True
678,679,0,3,43.0,1,6,46.9,2,True,False
737,738,1,1,35.0,0,0,512.3292,0,False,True
792,793,0,3,28.0,8,2,69.55,2,True,False
846,847,0,3,28.0,8,2,69.55,2,False,True
863,864,0,3,28.0,8,2,69.55,2,True,False


In [ ]:
# Normalize the featues
# split input variable from output variable
X = pdf[list(set(pdf.columns)-set(['Embarked']))]
y = pdf['Embarked']
y[:4]

,Embarked
0,2
1,0
2,2
3,2


In [ ]:
# Scale dataset using an standard scaler
scaler = sklearn.preprocessing.StandardScaler()

In [ ]:
# Normalize the feature
X = scaler.fit_transform(X)

In [ ]:
# split the dataset to training and testing dataset
X_train, x_test, Y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
X_train.shape, Y_train.shape, x_test.shape, y_test.shape

((712, 9), (712,), (179, 9), (179,))

**Train Random Forest**

In [ ]:
# define the random forest classifier model
rf = sklearn.ensemble.RandomForestClassifier(n_estimators=100)
rf

RandomForestClassifier()

In [ ]:
# Train the random forest classifier model
rf.fit(X=X_train, y=Y_train)

RandomForestClassifier()

In [ ]:
# Predict on the test dataset
y_pred = rf.predict(x_test)
y_pred

array([0, 2, 2, 2, 2, 0, 1, 2, 1, 2, 2, 2, 2, 0, 2, 2, 2, 1, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1,
       2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 1, 2,
       2, 0, 0, 0, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 0, 0, 2, 1, 0,
       2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 0, 2, 1, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0,
       2, 2, 2])

In [ ]:
# Check the accuracy score
score = sklearn.metrics.accuracy_score(y_test, y_pred)
score

0.7653631284916201

In [ ]:
conf_mat = sklearn.metrics.confusion_matrix(y_test, y_pred)
conf_mat

array([[ 14,   0,  29],
       [  0,  11,   6],
       [  5,   2, 112]])

In [ ]:
#### plotting logic
from sklearn import tree

In [ ]:
print(rf.feature_importnance_)
plt.figure()
tre.plot_tree(rf.estimators_[0], filled=True)
plt.show()

AttributeError: 'RandomForestClassifier' object has no attribute 'feature_importnance_'

In [ ]:
rf = RandomForestClassfier(max_dept=4, random_state=42)